# Exercise 11.2: Pileup reweighting in practice

From *Programming in High Energy Particle Physics*, Chapter 11

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch11/ex11_2_solution.ipynb)

Exercise 11.2 Pileup reweighting in practice Implement pileup reweighting for a simulated sample. Compare the \(n_{\text{vertices}}\) distribution before and after reweighting with data. Quantify the improvement with a \(\chi^2\) test for weighted histograms, and compare the means of the distributions.

<details><summary>Hint</summary>

Use Listing 11.2 with the pileup histogram for your certified luminosity sections. A Kolmogorov–Smirnov test is not appropriate here: it assumes unbinned, unweighted samples. For binned, weighted histograms use a \(\chi^2\) comparison that includes the sum of squared weights per bin, such as ROOT's TH1::Chi2Test with option "WW" (both histograms weighted) or "UW" (unweighted data vs. weighted MC), and normalize the MC to the data first. The number of reconstructed vertices is typically lower than the true number of interactions, so do not expect \(\langle n_{\text{vtx}}\rangle = \langle\mu\rangle\).

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q numpy scipy matplotlib")

**Data:** seeded toy pileup distributions stand in for a pileupCalc histogram and a simulated NanoAOD Pileup_nTrueInt branch. The data-like distribution is not treated as event-level truth.

### Step 1: Build certified-data and MC pileup profiles

In [ ]:
import numpy as np
rng=np.random.default_rng(42); N=100_000; edges=np.arange(0,101)
# Toy data pileup profile represents a certified-luminosity result, not measured truth in data.
mu_data=np.clip(rng.normal(32,7,N).astype(int),0,99)
mu_mc=np.clip(rng.normal(25,8,N).astype(int),0,99)
nv_data=rng.binomial(mu_data,.72); nv_mc=rng.binomial(mu_mc,.72)
data_pu,_=np.histogram(mu_data,bins=edges); mc_pu,_=np.histogram(mu_mc,bins=edges)
assert len(data_pu)==100

### Step 2: Reweight MC true interactions

In [ ]:
d=data_pu/data_pu.sum(); m=mc_pu/mc_pu.sum()
wbin=np.divide(d,m,out=np.zeros_like(d),where=m>0)
w=wbin[np.clip(mu_mc,0,99)]
assert np.isfinite(w).all(); print('Mean MC weight:',round(w.mean(),3))

### Step 3: Compare reconstructed vertex distributions with weighted chi-square

In [ ]:
from scipy.stats import chi2
import matplotlib.pyplot as plt
bins=np.arange(0,76)
obs,_=np.histogram(nv_data,bins=bins)
raw,_=np.histogram(nv_mc,bins=bins)
weighted,_=np.histogram(nv_mc,bins=bins,weights=w)
sumw2,_=np.histogram(nv_mc,bins=bins,weights=w*w)
weighted*=obs.sum()/weighted.sum(); sumw2*= (obs.sum()/np.sum(w))**2
raw_stat=np.sum((obs-raw)**2/np.maximum(obs+raw,1))
print('Raw chi2:',round(raw_stat,1))
valid=(obs+sumw2)>0
stat=np.sum((obs[valid]-weighted[valid])**2/(obs[valid]+sumw2[valid]))
ndf=int(valid.sum()-1); print(f'Weighted chi2/ndf = {stat:.1f}/{ndf}, p={chi2.sf(stat,ndf):.3g}')
print('Mean vertices: data',round(nv_data.mean(),2),'raw MC',round(nv_mc.mean(),2),'weighted MC',round(np.average(nv_mc,weights=w),2))
fig,ax=plt.subplots(figsize=(8,4)); centers=(bins[1:]+bins[:-1])/2
ax.step(centers,obs,where='mid',label='toy data'); ax.step(centers,raw,where='mid',label='raw MC'); ax.step(centers,weighted,where='mid',label='weighted MC')
ax.set(xlabel='Reconstructed vertices',ylabel='Events / vertex'); ax.legend(); plt.show()

Reweighting the simulated true interaction profile improves agreement in reconstructed vertices in this toy. The mean reconstructed vertex count remains below the mean number of true interactions. The chi-square includes data Poisson variance and MC sum of squared weights.